In [ ]:
import warnings
warnings.filterwarnings("ignore")

# Data preprocessing

In [ ]:
#Lectura de datos y escalado

import pandas as pd
import numpy as np

df = pd.read_excel("data_merged-3.xlsx")
glucose_data = df.iloc[:, 1:167] 

#Elegir entre escalar por columnas, filas o no escalar

scaling_method = 'row'  #A elegir

if scaling_method == 'row':
    #Por filas (por sujeto)
    glucose_scaled = glucose_data.apply( lambda row: (row - row.mean()) / row.std(ddof=0), axis=1 )
elif scaling_method == 'column':
    #Por columnas (por zonas)
    glucose_scaled = (glucose_data - glucose_data.mean()) / glucose_data.std(ddof=0)
elif scaling_method == 'none':
    glucose_scaled = glucose_data.copy()
else:
    raise ValueError("scaling_method must be 'row', 'column', or 'none'")

glucose_scaled = glucose_scaled.fillna(0)
df.iloc[:, 1:167] = glucose_scaled

print(df.head())


In [ ]:
#df = df.iloc[:-4, :] #quitamos sujetos sin datos
#df = df.drop([3, 17, 25, 27]) #quitamos fenotipo 1 y 4
df = df.iloc[:, 1:] #quitamos etiqueta
phenotype = df.iloc[:, 166] 
glucose_scaled = df.iloc[:, 0:166]
phenotype

In [ ]:
from scipy.stats import shapiro

non_normal = 0
for j in range(glucose_scaled.shape[1]):
    stat, p = shapiro(glucose_scaled.iloc[:, j])
    if p < 0.05:
        non_normal += 1

print(f"Non-normal zones: {non_normal} of {glucose_data.shape[1]}")


In [ ]:
# ANAMORFOSIS NO LINEAL SOBRE LAS 15 ZONAS NO NORMALES

import numpy as np
import pandas as pd
from scipy.stats import norm, rankdata, shapiro
import matplotlib.pyplot as plt
import seaborn as sns

# --- Nonlinear anamorphosis for a single column ---
def nonlinear_anamorphosis_col(col):
    ranks = rankdata(col)
    uniform = (ranks - 0.5) / len(ranks)
    return norm.ppf(uniform)

# --- Copy your already row-scaled glucose data ---
glucose_scaled_ana = glucose_scaled.copy()

# --- Identify non-normal zones ---
non_normal_zones = []
for zone in glucose_scaled.columns:
    stat, p = shapiro(glucose_scaled[zone])
    if p < 0.05:
        non_normal_zones.append(zone)

print(f"Non-normal zones: {len(non_normal_zones)}")
print("Zones:", non_normal_zones)

# --- Apply nonlinear anamorphosis only to those zones ---
for zone in non_normal_zones:
    glucose_scaled_ana[zone] = nonlinear_anamorphosis_col(glucose_scaled[zone])

# --- Check improvement in skewness ---
print("Original skewness (mean):", glucose_scaled.skew().mean())
print("After selective anamorphosis (mean):", glucose_scaled_ana.skew().mean())


sns.set_theme(style="whitegrid", rc={
    "axes.labelsize": 12,
    "axes.titlesize": 14,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "figure.figsize": (11, 7)
})

zones_to_plot = np.random.choice(non_normal_zones, min(3, len(non_normal_zones)), replace=False)

fig, axes = plt.subplots(len(zones_to_plot), 2, figsize=(12, 4 * len(zones_to_plot)))

for i, zone in enumerate(zones_to_plot):

    # Before
    sns.histplot(
        glucose_scaled[zone], 
        bins=14, kde=True, ax=axes[i, 0],
        color="#4C72B0", edgecolor="white", alpha=0.85
    )
    axes[i, 0].set_title(f"{zone} — Antes de anamorfosis")
    axes[i, 0].set_xlabel("Valor de glucosa (estandarizado)")
    axes[i, 0].set_ylabel("Frecuencia")

    # After
    sns.histplot(
        glucose_scaled_ana[zone], 
        bins=14, kde=True, ax=axes[i, 1],
        color="#DD8452", edgecolor="white", alpha=0.85
    )
    axes[i, 1].set_title(f"{zone} — Después de anamorfosis")
    axes[i, 1].set_xlabel("Valor transformado")
    axes[i, 1].set_ylabel("Frecuencia")

plt.tight_layout()
plt.savefig("anamorphosis_comparison_TFM.png", dpi=300, bbox_inches="tight")
plt.show()

# --- Update dataframe ---
df.iloc[:, 0:166] = glucose_scaled_ana


# Brain regions relevant to clinical phenotype

Kruskal–Wallis and Fisher ratio are used to identify relevant regions.

In [ ]:
#Kruskal top zones
import pandas as pd
import scipy.stats as stats

results = {}
for col in df.columns[:166]:  #Las 166 zonas de glucosa
    groups = [df[col][phenotype == g].dropna() for g in phenotype.unique()]
    
    # Only run if the column haas variability
    if all(len(g) > 0 for g in groups):
        combined = pd.concat(groups)
        if combined.nunique() > 1:  # skip if all values identical
            try:
                stat, p = stats.kruskal(*groups)
                results[col] = p
            except ValueError:
                continue

# Collect results in a dataframe
kruskal_df = pd.DataFrame.from_dict(results, orient="index", columns=["p_value"])
kruskal_df = kruskal_df.sort_values("p_value")

print(kruskal_df.head(23))
top_zones = kruskal_df.head(10).index
top_glucose_data = df[top_zones]

In [ ]:
#Fisher ratio top zones
results = {}
for col in df.columns[:166]:  # first 166 glucose zones
    # Use df['Fenotipo'] as the grouping column
    groups = [df[col][phenotype == g].dropna() for g in phenotype.unique()]
    
    # Only run if the column has variability
    if all(len(g) > 0 for g in groups):
        combined = pd.concat(groups)
        if combined.nunique() > 1:  # skip if all values identical
            # Compute Fisher ratio: between-group variance / within-group variance
            overall_mean = combined.mean()
            between = sum(len(g) * (g.mean() - overall_mean)**2 for g in groups)
            within = sum(((g - g.mean())**2).sum() for g in groups)
            F = between / within if within > 0 else np.nan
            results[col] = F

# Collect results in a dataframe
fisher_df = pd.DataFrame.from_dict(results, orient="index", columns=["FisherRatio"])
fisher_df = fisher_df.sort_values("FisherRatio", ascending=False)

print(fisher_df.head(15))  # top 15 most discriminative zones
top_zones_fisher = fisher_df.head(10).index
top_glucose_data_fisher = df[top_zones_fisher]


In [ ]:
#Mostrar kruskal + fisher
import pandas as pd
import matplotlib.pyplot as plt

# Take top 20 zones according to each method
top_kruskal = kruskal_df.head(20).copy()
top_fisher = fisher_df.head(20).copy()

# Merge for comparison (inner join on zone name)
comparison_df = pd.merge(top_kruskal, top_fisher, left_index=True, right_index=True, how='outer')
comparison_df = comparison_df.sort_values("p_value")

print("Top zones according to Kruskal-Wallis and Fisher Ratio:")
print(comparison_df.round(4))

# --- Optional: visualize top zones ---
plt.figure(figsize=(12,6))

# Bar plot for Kruskal p-values
plt.subplot(1,2,1)
plt.barh(top_kruskal.index.astype(str), -np.log10(top_kruskal['p_value']), color='skyblue')
plt.xlabel('-log10(p-value)')
plt.title('Top 20 zonas por Kruskal-Wallis')

# Bar plot for Fisher ratio
plt.subplot(1,2,2)
plt.barh(top_fisher.index.astype(str), top_fisher['FisherRatio'], color='salmon')
plt.xlabel('Fisher Ratio')
plt.title('Top 20 zonas por Fisher Ratio')

plt.tight_layout()
plt.savefig("kruskalfisher.png", dpi=300, bbox_inches="tight")  # PNG format
plt.show()


In [ ]:
import numpy as np

# Invert p-value so that lower p → higher score
kw_score = -np.log10(kruskal_df['p_value'])

# Normalize both to 0-1 range
kw_norm = (kw_score - kw_score.min()) / (kw_score.max() - kw_score.min())
fisher_norm = (fisher_df['FisherRatio'] - fisher_df['FisherRatio'].min()) / (fisher_df['FisherRatio'].max() - fisher_df['FisherRatio'].min())
combined_score = 0.5 * kw_norm + 0.5 * fisher_norm  # example weights
combined_df = pd.DataFrame({
    'KruskalNorm': kw_norm,
    'FisherNorm': fisher_norm,
    'CombinedScore': combined_score
})

combined_df = combined_df.sort_values('CombinedScore', ascending=False)
top_zones = combined_df.head(20)
print(top_zones.round(4))

plt.figure(figsize=(8,6))
plt.barh(top_zones.index.astype(str), top_zones['CombinedScore'], color='purple')
plt.xlabel('Puntuación según criterio combinado')
plt.title('Top 20 Zonas más relevantes (Criterio combinado)')
plt.gca().invert_yaxis()
plt.savefig("kruskalfisher11.png", dpi=300, bbox_inches="tight")  # PNG format
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score
import pandas as pd

# --- Compute normalized Kruskal and Fisher scores ---
kw_score = -np.log10(kruskal_df['p_value'])
kw_norm = (kw_score - kw_score.min()) / (kw_score.max() - kw_score.min())
fisher_norm = (fisher_df['FisherRatio'] - fisher_df['FisherRatio'].min()) / (fisher_df['FisherRatio'].max() - fisher_df['FisherRatio'].min())

# --- Combined weighted score ---
combined_score = 0.5 * kw_norm + 0.5 * fisher_norm
combined_df = pd.DataFrame({
    'KruskalNorm': kw_norm,
    'FisherNorm': fisher_norm,
    'CombinedScore': combined_score
}).sort_values('CombinedScore', ascending=False)

# --- Range of top zones to test ---
top_range = range(2, 23, 1)
ari_kw_list, ari_fisher_list, ari_combined_list = [], [], []

n_clusters = len(phenotype.unique())

for top_n in top_range:
    # Select top zones for each method
    kw_top_zones = kruskal_df.head(top_n).index
    fisher_top_zones = fisher_df.head(top_n).index
    combined_top_zones = combined_df.head(top_n).index

    # Get corresponding data
    glucose_kw = df[kw_top_zones]
    glucose_fisher = df[fisher_top_zones]
    glucose_combined = df[combined_top_zones]

    # --- PCA + KMeans + ARI ---
    X_kw = PCA(n_components=2).fit_transform(glucose_kw)
    clusters_kw = KMeans(n_clusters=n_clusters, random_state=42).fit_predict(X_kw)
    ari_kw = adjusted_rand_score(phenotype, clusters_kw)
    ari_kw_list.append(ari_kw)

    X_fisher = PCA(n_components=2).fit_transform(glucose_fisher)
    clusters_fisher = KMeans(n_clusters=n_clusters, random_state=42).fit_predict(X_fisher)
    ari_fisher = adjusted_rand_score(phenotype, clusters_fisher)
    ari_fisher_list.append(ari_fisher)

    X_combined = PCA(n_components=2).fit_transform(glucose_combined)
    clusters_combined = KMeans(n_clusters=n_clusters, random_state=42).fit_predict(X_combined)
    ari_combined = adjusted_rand_score(phenotype, clusters_combined)
    ari_combined_list.append(ari_combined)

# --- Plot all results together ---
plt.figure(figsize=(8,5))
plt.plot(top_range, ari_kw_list, marker='o', color='steelblue', label='Kruskal-Wallis top zones')
plt.plot(top_range, ari_fisher_list, marker='s', color='tomato', label='Fisher Ratio top zones')
plt.plot(top_range, ari_combined_list, marker='^', color='purple', label='Combined score top zones')

plt.xlabel('Number of top zones used', fontsize=12)
plt.ylabel('Adjusted Rand Index (ARI)', fontsize=12)
plt.title('Clustering performance vs number of top zones', fontsize=14, fontweight='bold')
plt.xticks(top_range)
plt.ylim(-0.1, 1)
plt.legend()
plt.grid(True, linestyle='--', alpha=0.4)

plt.tight_layout()
plt.savefig("topzones_all_methods.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
#Comparación de la mejor agrupación kruskal top zones (19) con la de fisher ratio (11)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score

# --- Parameters ---
n_clusters = len(phenotype.unique())  # number of phenotype groups

# --- Select top zones ---
kw_top_zones = kruskal_df.head(4).index
fisher_top_zones = fisher_df.head(2).index

glucose_kw = df[kw_top_zones]
glucose_fisher = df[fisher_top_zones]

# --- Function to run PCA + KMeans + ARI ---
def run_pca_kmeans(glucose_data, phenotype_series, n_clusters):
    pca = PCA(n_components=2)
    X_pca = pca.fit_transform(glucose_data)
    
    kmeans = KMeans(n_clusters=n_clusters, random_state=42)
    clusters = kmeans.fit_predict(X_pca)
    
    ari = adjusted_rand_score(phenotype_series, clusters)
    
    return X_pca, clusters, ari

# --- Run both ---
X_kw, clusters_kw, ari_kw = run_pca_kmeans(glucose_kw, phenotype, n_clusters)
X_fisher, clusters_fisher, ari_fisher = run_pca_kmeans(glucose_fisher, phenotype, n_clusters)

# --- Plot side by side ---
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# KW top zones
scatter1 = axes[0].scatter(X_kw[:,0], X_kw[:,1], c=clusters_kw, cmap='tab10', s=50)
for i, txt in enumerate(phenotype):
    axes[0].annotate(str(int(txt)), (X_kw[i,0]+0.02, X_kw[i,1]+0.02), fontsize=9)
axes[0].set_title(f'KW top zones (10), ARI={ari_kw:.3f}')
axes[0].set_xlabel('PC1')
axes[0].set_ylabel('PC2')
axes[0].legend(*scatter1.legend_elements(), title="Clusters")

# Fisher top zones
scatter2 = axes[1].scatter(X_fisher[:,0], X_fisher[:,1], c=clusters_fisher, cmap='tab10', s=50)
for i, txt in enumerate(phenotype):
    axes[1].annotate(str(int(txt)), (X_fisher[i,0]+0.02, X_fisher[i,1]+0.02), fontsize=9)
axes[1].set_title(f'Fisher top zones (4), ARI={ari_fisher:.3f}')
axes[1].set_xlabel('PC1')
axes[1].set_ylabel('PC2')
axes[1].legend(*scatter2.legend_elements(), title="Clusters")

plt.tight_layout()
plt.savefig("clus.png", dpi=300, bbox_inches="tight")  # PNG format
plt.show()


print("KW top zones ARI:", round(ari_kw, 3))
print(pd.crosstab(phenotype, clusters_kw).round(0))

print("\nFisher top zones ARI:", round(ari_fisher, 3))
print(pd.crosstab(phenotype, clusters_fisher).round(0))

# Crosstabs
ct_kw = pd.crosstab(phenotype, clusters_kw)
ct_fisher = pd.crosstab(phenotype, clusters_fisher)

# Create side-by-side bar plots
fig, axes = plt.subplots(1, 2, figsize=(14,6))

# KW top zones
ct_kw.T.plot(kind='bar', ax=axes[0], colormap='tab10')
axes[0].set_xlabel('Cluster')
axes[0].set_ylabel('Número de sujetos')
axes[0].set_title('KW top zones: Fenotipo por cluster')
axes[0].legend(title='Fenotipo')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=0)

# Fisher top zones
ct_fisher.T.plot(kind='bar', ax=axes[1], colormap='tab10')
axes[1].set_xlabel('Cluster')
axes[1].set_ylabel('Número de sujetos')
axes[1].set_title('Fisher top zones: Fenotipo por cluster')
axes[1].legend(title='Fenotipo')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=0)

plt.tight_layout()
plt.savefig("bar.png", dpi=300, bbox_inches="tight")  # PNG format
plt.show()

In [ ]:
#Mejor agrupación

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score

# --- Parameters ---
n_clusters = len(phenotype.unique())  # number of phenotype groups

# --- Select top Kruskal-Wallis zones ---
kw_top_zones = fisher_df.head(2).index
glucose_kw = df[kw_top_zones]

# --- Function to run PCA + KMeans + ARI ---
def run_pca_kmeans(glucose_data, phenotype_series, n_clusters):
    pca = PCA(n_components=2)
    X_pca = pca.fit_transform(glucose_data)
    
    kmeans = KMeans(n_clusters=n_clusters, random_state=42)
    clusters = kmeans.fit_predict(X_pca)
    
    ari = adjusted_rand_score(phenotype_series, clusters)
    
    return X_pca, clusters, ari

# --- Run KW ---
X_kw, clusters_kw, ari_kw = run_pca_kmeans(glucose_kw, phenotype, n_clusters)

# --- Scatter plot for KW ---
plt.figure(figsize=(8,6))
scatter = plt.scatter(X_kw[:,0], X_kw[:,1], c=clusters_kw, cmap='tab10', s=50)
for i, txt in enumerate(phenotype):
    plt.annotate(str(int(txt)), (X_kw[i,0]+0.02, X_kw[i,1]+0.02), fontsize=9)
plt.title(f'Fisher Ratio top zonas (2), ARI={ari_kw:.3f}')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.legend(*scatter.legend_elements(), title="Clusters")
plt.tight_layout()
plt.savefig("kw_clusters.png", dpi=300, bbox_inches="tight")
plt.show()

# --- Crosstab and bar plot ---
ct_kw = pd.crosstab(phenotype, clusters_kw)
print("Fisher Ratio top zones ARI:", round(ari_kw, 3))
print(ct_kw.round(0))



In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score
import seaborn as sns

sns.set_theme(style="whitegrid", rc={
    "axes.titlesize": 14,
    "axes.labelsize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
    "figure.figsize": (8,6)
})

# --- Parámetros ---
n_clusters = len(phenotype.unique())

# --- Seleccionar las top zonas ---
top_zones = fisher_df.head(2).index
glucose_top = df[top_zones]

# --- Función PCA + KMeans + ARI ---
def run_pca_kmeans(glucose_data, phenotype_series, n_clusters):
    pca = PCA(n_components=2)
    X_pca = pca.fit_transform(glucose_data)
    
    kmeans = KMeans(n_clusters=n_clusters, random_state=42)
    clusters = kmeans.fit_predict(X_pca)
    
    # Invertir etiquetas: 0→1, 1→0 (solo si n_clusters=2)
    if n_clusters == 2:
        clusters = 1 - clusters
    
    ari = adjusted_rand_score(phenotype_series, clusters)
    
    return X_pca, clusters, ari, kmeans.cluster_centers_

# --- Ejecutar ---
X_pca, clusters, ari, centers = run_pca_kmeans(glucose_top, phenotype, n_clusters)

# --- Gráfico ---
plt.figure(figsize=(8,6))
palette = sns.color_palette("tab10", n_clusters)
for i in range(n_clusters):
    plt.scatter(
        X_pca[clusters==i,0], 
        X_pca[clusters==i,1], 
        s=80, 
        alpha=0.8, 
        label=f"Cluster {i+1}", 
        color=palette[i],
        edgecolor='k'
    )

# Centros de cluster
plt.scatter(
    centers[:,0], centers[:,1],
    c='black', s=100, marker='X', label='Centros'
)

# Anotaciones con fenotipo
for i, txt in enumerate(phenotype):
    plt.text(X_pca[i,0]+0.02, X_pca[i,1]+0.02, str(int(txt)), fontsize=9)

# Etiquetas y títulos
plt.title(f'Top zonas por Fisher Ratio (n={len(top_zones)}), ARI={ari:.3f}', fontsize=14, weight='bold')
plt.xlabel('PC1', fontsize=12)
plt.ylabel('PC2', fontsize=12)
plt.legend(title="Clusters", loc='best')
plt.grid(True, linestyle='--', alpha=0.3)
sns.despine(trim=True)

plt.tight_layout()
plt.savefig("top_zones_clusters_TFM_invert.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# --- Step 1: PCA on the data ---
kw_top_zones = fisher_df.head(2).index

# top_zones: list of the top glucose columns from Kruskal-Wallis
X = df[kw_top_zones]  # already scaled


pca = PCA(n_components=2)
pcs = pca.fit_transform(X)

# --- KMeans clustering with k=2 using 2 PCs ---
k = 2
kmeans = KMeans(n_clusters=k, random_state=123)
clusters = kmeans.fit_predict(pcs)

# --- Create DataFrame with PCs and cluster labels ---
pc_df = pd.DataFrame(pcs, columns=['PC1', 'PC2'], index=glucose_scaled_ana.index)
pc_df['Cluster'] = clusters.astype(str)

# --- Plot (PC1 vs PC2) ---
plt.figure(figsize=(8,6))
sns.scatterplot(
    data=pc_df,
    x='PC1',
    y='PC2',
    hue='Cluster',
    palette='Set1',
    s=100,
    alpha=0.8,
    edgecolor='k'
)

# Plot cluster centers
centers = kmeans.cluster_centers_
plt.scatter(
    centers[:,0], centers[:,1],
    c='black',
    s=60,
    marker='s',
    label='Centers'
)

# Titles and labels
plt.title("2D PCA (Fisher Ratio 2 top zones) with KMeans Clusters (k=2)", fontsize=14, weight='bold')
plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)", fontsize=12)
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)", fontsize=12)
plt.legend(title='Cluster')
plt.grid(True, linestyle='--', alpha=0.3)
sns.despine(trim=True)

# Save figure
plt.tight_layout()
plt.savefig("pca_kmeans_2D1.png", dpi=300, bbox_inches="tight")
plt.show()

# Note: Using PC1+PC2 or PC1+PC2+PC3 produces very similar clusters.
# We will proceed with this 2D version for simplicity.


In [ ]:
#Miramos las demás variables según clustering de fisher top zonas 2
import pandas as pd
import scipy.stats as stats

df['Cluster'] = kmeans.labels_  
df = df.reset_index(drop=True) 
df = df.rename(columns={df.columns[166]: 'Fenotipo'})
pc_df = pc_df.reset_index(drop=True)

results = {}
for col in df.columns[166:229]:  # all variables after the 166 glucose zones
    # Create groups according to cluster
    groups = [df[col][df['Cluster'] == g].dropna() for g in df['Cluster'].unique()]
    
    try:
        stat, p = stats.kruskal(*groups)
        results[col] = {"H_stat": stat, "p_value": p}
    except ValueError:
        continue

# Collect results in a DataFrame
kruskal_other = pd.DataFrame(results).T
kruskal_other = kruskal_other.sort_values("p_value")
pd.set_option('display.float_format', '{:f}'.format)
print(kruskal_other.head(30))


In [ ]:
# ================================
# Miramos las demás variables según clustering usando Chi-cuadrado
# ================================
import pandas as pd
import scipy.stats as stats

# Aseguramos que la columna Cluster existe
df['Cluster'] = kmeans.labels_
df = df.reset_index(drop=True)
df = df.rename(columns={df.columns[166]: 'Fenotipo'})  # ajusta según tu índice
pc_df = pc_df.reset_index(drop=True)

chi2_results = {}

# Recorremos las variables categóricas (por ejemplo, de 166 a 228)
for col in df.columns[166:229]:
    # Creamos la tabla de contingencia: Cluster x Variable
    contingency = pd.crosstab(df['Cluster'], df[col])
    
    # Solo ejecutamos si la tabla tiene al menos 2 filas y 2 columnas
    if contingency.shape[0] > 1 and contingency.shape[1] > 1:
        chi2, p, dof, expected = stats.chi2_contingency(contingency)
        chi2_results[col] = {"Chi2_stat": chi2, "p_value": p, "dof": dof}

# Guardamos en DataFrame
chi2_other = pd.DataFrame(chi2_results).T
chi2_other = chi2_other.sort_values("p_value")

pd.set_option('display.float_format', '{:f}'.format)
print(chi2_other.head(30))


In [ ]:
# Add cluster labels to top_glucose_data
top_glucose_data_with_clusters = top_glucose_data.copy()
top_glucose_data_with_clusters["Cluster"] = clusters

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Make sure to exclude the Cluster column explicitly
zones = [col for col in top_glucose_data_with_clusters.columns if col != "Cluster"]
x = np.arange(len(zones))
width = 0.35  # width of each bar

# Colormap for clusters
colors = plt.cm.Set2(np.linspace(0, 1, len(cluster_means.index)))

fig, ax = plt.subplots(figsize=(12,6))

for i, (cluster, color) in enumerate(zip(cluster_means.index, colors)):
    ax.bar(
        x + i*width, 
        cluster_means.loc[cluster, zones], 
        width, 
        yerr=cluster_stds.loc[cluster, zones], 
        capsize=4, 
        color=color, 
        edgecolor="black", 
        linewidth=0.6,
        label=f'Cluster {cluster}'
    )

# Beautify the plot
ax.set_xticks(x + width/2)
ax.set_xticklabels(zones, rotation=45, ha="right")
ax.set_ylabel('Glucosa escalada', fontsize=12)
ax.set_title('Glucosa media ± SD por zona y cluster', fontsize=14, fontweight="bold")
ax.legend(title="Clusters")
ax.grid(axis="y", linestyle="--", alpha=0.6)

plt.tight_layout()
plt.savefig("dif_nicer.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
#Valores medios de glucosa en top zones por fenotipo


import matplotlib.pyplot as plt
import numpy as np

# --- Calcular medias y desviaciones estándar por fenotipo ---
phenotype_means = df.groupby(phenotype).mean()
phenotype_stds = df.groupby(phenotype).std()

# --- Seleccionar las zonas relevantes (como las top del Kruskal-Wallis) ---
zones = [col for col in top_glucose_data_with_clusters.columns if col != "Cluster"]
x = np.arange(len(zones))
width = 0.35  # ancho de cada barra

# --- Colores para cada fenotipo ---
colors = plt.cm.Set1(np.linspace(0, 1, len(phenotype_means.index)))

# --- Graficar ---
fig, ax = plt.subplots(figsize=(12, 6))

for i, (pheno, color) in enumerate(zip(phenotype_means.index, colors)):
    ax.bar(
        x + i*width,
        phenotype_means.loc[pheno, zones],
        width,
        yerr=phenotype_stds.loc[pheno, zones],
        capsize=4,
        color=color,
        edgecolor="black",
        linewidth=0.6,
        label=f'Fenotipo {int(pheno)}'

    )

# --- Decoración ---
ax.set_xticks(x + width/2)
ax.set_xticklabels(zones, rotation=45, ha="right")
ax.set_ylabel('Glucosa escalada', fontsize=12)
ax.set_title('Glucosa media ± SD por zona y fenotipo', fontsize=14, fontweight="bold")
ax.legend(title="Fenotipos")
ax.grid(axis="y", linestyle="--", alpha=0.6)

plt.tight_layout()
plt.savefig("glucosa_fenotipos.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
results = {}

# Support Vector Machines (SVM)

Hyperparameter optimization and comparison of sigmoid and RBF kernels.

# Sigmoid SVM hyperparameter optimization

In [ ]:

from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.svm import SVC
from sklearn.decomposition import PCA
from collections import Counter, defaultdict
import numpy as np

# --- Parameter grid for SVM ---
param_grid = {
    'C': np.linspace(5, 15, 6),
    'gamma': ['scale', 'auto', 0.2] ,
    'coef0': np.linspace(0, 0.3, 5),
    'class_weight': [{2: 1, 3: 1.2}]  
}



#np.logspace(-2, 2, 5)
#np.linspace(0.05, 0.3, 6)



# --- Range of top Kruskal–Wallis zones and PCA components ---
zone_range = range(2,3)   
pca_range = range(2,3)      

# --- Store results ---
results = []

for n_zones in zone_range:
    kw_top_zones = fisher_df.head(n_zones).index
    X = df[kw_top_zones].values
    y = phenotype.values

    for n_comp in pca_range:
        # --- PCA ---
        pca = PCA(n_components=n_comp)
        X_pca = pca.fit_transform(X)

        for seed in range(100):  # number of random seeds
            cv = StratifiedKFold(n_splits=7, shuffle=True, random_state=seed)
            svm = SVC(kernel='sigmoid', random_state=seed)
            grid = GridSearchCV(
                svm, param_grid, cv=cv,
                scoring='f1_macro',  # optimize macro F1
                n_jobs=-1
            )
            grid.fit(X_pca, y)
            results.append((n_zones, n_comp, seed, grid.best_params_, grid.best_score_))

# --- Aggregate results across seeds ---
config_scores = defaultdict(list)
for n_zones, n_comp, seed, params, f1 in results:
    key = (n_zones, n_comp, str(params))
    config_scores[key].append(f1)

# --- Compute mean ± std for each configuration ---
config_summary = []
for (n_zones, n_comp, params_str), f1_list in config_scores.items():
    mean_f1 = np.mean(f1_list)
    std_f1 = np.std(f1_list)
    config_summary.append((mean_f1, std_f1, n_zones, n_comp, params_str, len(f1_list)))

# --- Sort by mean F1 descending ---
config_summary.sort(reverse=True, key=lambda x: x[0])

# --- Print top 5 configurations ---
print("\n=== Top SVM (Sigmoid) Configurations (Aggregated over seeds) ===")
for mean_f1, std_f1, n_zones, n_comp, params_str, n_runs in config_summary[:10]:
    print(f"Top {n_zones:2d} zones | PCA {n_comp} | F1-macro = {mean_f1:.3f} ± {std_f1:.3f} "
          f"({n_runs} seeds) | Params: {params_str}")

# --- Most frequent best parameter sets ---
best_params_all = [r[3] for r in results]
param_counts = Counter(map(str, best_params_all))

print("\nParameter frequency across all configurations:")
for p, count in param_counts.most_common():
    print(f"{count} occurrences → {p}")


In [ ]:
# === SVM (Sigmoid) PERFORMANCE EVALUATION – FINAL FIXED VERSION ===

from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.svm import SVC
from sklearn.decomposition import PCA
from sklearn.metrics import (
    confusion_matrix, classification_report, roc_auc_score,
    average_precision_score, precision_recall_curve, roc_curve
)
import numpy as np
import matplotlib.pyplot as plt

# --- Data ---
kw_top_zones = kruskal_df.head(20).index
X = df[kw_top_zones].values
y = phenotype.values   # now values are 1 and 2

# --- PCA ---
pca = PCA(n_components=4)
X_pca = pca.fit_transform(X)

# --- Best SVM parameters from grid search ---
svm_params = {
    'C': 10,
    'gamma': 0.2,
    'coef0': 0,
    'kernel': 'sigmoid',
    'probability': True  # needed for predict_proba
}

# --- Fit once to know class order ---
tmp_model = SVC(**svm_params)
tmp_model.fit(X_pca, y)
idx_class2 = np.where(tmp_model.classes_ == 2)[0][0]   # now class = 2 instead of 3

# --- Initialize accumulators ---
n_runs = 50
accs, aucs, pr_aucs, f1_macros, f1_weighted = [], [], [], [], []
cm_total = np.zeros((2, 2))
phen2_precision, phen2_recall, phen2_f1 = [], [], []

# --- Collect ROC/PR data for averaged curves ---
fpr_all, tpr_all, prec_all, rec_all = [], [], [], []

# --- Loop over random seeds ---
for seed in range(n_runs):
    cv = StratifiedKFold(n_splits=7, shuffle=True, random_state=seed)
    svm = SVC(**svm_params)

    # Cross-validated predictions (out-of-fold)
    y_pred = cross_val_predict(svm, X_pca, y, cv=cv)
    y_prob = cross_val_predict(svm, X_pca, y, cv=cv, method='predict_proba')
    y_prob_phen2 = y_prob[:, idx_class2]

    # --- Metrics per run ---
    accs.append(np.mean(y_pred == y))

    # AUC – no pos_label needed because we pass prob(class2)
    aucs.append(roc_auc_score(y, y_prob_phen2))

    # PR-AUC – same, prob(class2) = positive class
    pr_aucs.append(average_precision_score(y, y_prob_phen2, pos_label=2))

    cm_total += confusion_matrix(y, y_pred, labels=np.unique(y))

    report = classification_report(y, y_pred, output_dict=True)
    f1_macros.append(report['macro avg']['f1-score'])
    f1_weighted.append(report['weighted avg']['f1-score'])
    phen2_precision.append(report['2']['precision'])
    phen2_recall.append(report['2']['recall'])
    phen2_f1.append(report['2']['f1-score'])

    # --- ROC & PR for averaging ---
    fpr, tpr, _ = roc_curve(y, y_prob_phen2, pos_label=2)
    prec, rec, _ = precision_recall_curve(y, y_prob_phen2, pos_label=2)

    fpr_all.append(np.interp(np.linspace(0,1,100), fpr, tpr))
    tpr_all.append(np.interp(np.linspace(0,1,100), fpr, tpr))
    prec_all.append(np.interp(np.linspace(0,1,100), rec[::-1], prec[::-1]))
    rec_all.append(np.linspace(0,1,100))

# --- Summary ---
print("\n=== SVM (Sigmoid) Extended Evaluation (Cross-Validated) ===")
print(f"Accuracy: {np.mean(accs):.3f} ± {np.std(accs):.3f}")
print(f"ROC AUC: {np.mean(aucs):.3f} ± {np.std(aucs):.3f}")
print(f"PR AUC: {np.mean(pr_aucs):.3f} ± {np.std(pr_aucs):.3f}")
print(f"F1 Macro: {np.mean(f1_macros):.3f} ± {np.std(f1_macros):.3f}")
print(f"F1 Weighted: {np.mean(f1_weighted):.3f} ± {np.std(f1_weighted):.3f}")
print(f"Phenotype 2 → Precision: {np.mean(phen2_precision):.3f}, "
      f"Recall: {np.mean(phen2_recall):.3f}, F1: {np.mean(phen2_f1):.3f}")

print("\nMean confusion matrix:\n", cm_total / n_runs)

# --- Mean ROC and PR curves across seeds ---
mean_tpr = np.mean(tpr_all, axis=0)
mean_prec = np.mean(prec_all, axis=0)
mean_rec = np.mean(rec_all, axis=0)

results['SVM (Sigmoid)'] = {
    'tpr': mean_tpr,
    'auc': np.mean(aucs),
    'prec': mean_prec,
    'rec': mean_rec,
    'ap': np.mean(pr_aucs)
}

plt.figure(figsize=(12,5))

plt.subplot(1,2,1)
plt.plot(np.linspace(0,1,100), mean_tpr, label=f"Mean AUC = {np.mean(aucs):.3f}")
plt.plot([0,1],[0,1],'--',color='gray')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Mean ROC Curve (Phenotype 2)")
plt.legend()

plt.subplot(1,2,2)
plt.plot(mean_rec, mean_prec, label=f"Mean AP = {np.mean(pr_aucs):.3f}")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Mean Precision–Recall Curve (Phenotype 2)")
plt.legend()

plt.tight_layout()
plt.show()


In [ ]:
# === SVM (Sigmoid) BOOTSTRAP EVALUATION – FULLY STABILIZED ===

from sklearn.utils import resample
from sklearn.svm import SVC
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, average_precision_score
import numpy as np

# --- Data ---
kw_top_zones = kruskal_df.head(20).index
X = df[kw_top_zones].values
y = phenotype.values   # now values are 1 and 2

# --- PCA ---
pca = PCA(n_components=4)
X_pca = pca.fit_transform(X)

# --- Best SVM parameters ---
svm_params = {
    'C': 10,
    'gamma': 0.2,
    'coef0': 0,
    'kernel': 'sigmoid',
    'probability': True
}

# --- Bootstrap ---
n_bootstrap = 1000
min_oob_size = 5
accs_bs, f1_macro_bs, f1_weighted_bs, auc_bs, pr_auc_bs = [], [], [], [], []

# Fit once to get class index for phenotype 2
tmp_model = SVC(**svm_params)
tmp_model.fit(X_pca, y)
idx_class2 = np.where(tmp_model.classes_ == 2)[0][0]

for i in range(n_bootstrap):
    # Resample training indices
    train_idx = np.unique(resample(np.arange(len(X_pca)), replace=True, stratify=y))
    X_res, y_res = X_pca[train_idx], y[train_idx]
    
    # Fit model on bootstrap sample
    model = SVC(**svm_params)
    model.fit(X_res, y_res)
    
    # OOB samples = complement of training indices
    oob_mask = ~np.isin(np.arange(len(X_pca)), train_idx)
    X_oob = X_pca[oob_mask]
    y_oob = y[oob_mask]
    
    # Skip iteration if OOB too small or only one class
    if len(y_oob) < min_oob_size:
        continue
    if len(np.unique(y_oob)) < 2:
        continue
    
    # Evaluate
    y_pred = model.predict(X_oob)
    y_prob = model.predict_proba(X_oob)[:, idx_class2]   # probability of class 2
    
    accs_bs.append(accuracy_score(y_oob, y_pred))
    f1_macro_bs.append(f1_score(y_oob, y_pred, average='macro'))
    f1_weighted_bs.append(f1_score(y_oob, y_pred, average='weighted'))

    # ROC AUC → no pos_label needed because prob = P(class2)
    auc_bs.append(roc_auc_score(y_oob, y_prob))

    # PR AUC
    pr_auc_bs.append(average_precision_score(y_oob, y_prob))
    

# --- Compute 95% confidence intervals ---
def ci95(arr):
    return np.percentile(arr, 2.5), np.percentile(arr, 97.5)

print("=== SVM (Sigmoid) Metrics with 95% CI (OOB, Fully Stabilized) ===")
print(f"Accuracy: {np.mean(accs_bs):.3f} (95% CI {ci95(accs_bs)})")
print(f"F1 Macro: {np.mean(f1_macro_bs):.3f} (95% CI {ci95(f1_macro_bs)})")
print(f"F1 Weighted: {np.mean(f1_weighted_bs):.3f} (95% CI {ci95(f1_weighted_bs)})")
print(f"ROC AUC: {np.mean(auc_bs):.3f} (95% CI {ci95(auc_bs)})")
print(f"PR AUC: {np.mean(pr_auc_bs):.3f} (95% CI {ci95(pr_auc_bs)})")


# RBF SVM hyperparameter optimization

In [ ]:
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.svm import SVC
from sklearn.decomposition import PCA
from collections import Counter, defaultdict
import numpy as np

# --- Parameter grid for SVM (RBF kernel) ---
param_grid = {
    'C': [10,12,8],
    'gamma': ['scale', 0.1, 0.2],
    'class_weight': [{2: 1, 3: 1.2}]
}

# --- Range of top Kruskal–Wallis zones and PCA components ---
zone_range = range(2, 5)   # top 18–28 zones
pca_range = range(2, 3)      # PCA components: 2–4

# --- Store results ---
results = []

for n_zones in zone_range:
    kw_top_zones = fisher_df.head(n_zones).index
    X = df[kw_top_zones].values
    y = phenotype.values

    for n_comp in pca_range:
        # --- PCA ---
        pca = PCA(n_components=n_comp)
        X_pca = pca.fit_transform(X)

        for seed in range(50):  # number of random seeds
            cv = StratifiedKFold(n_splits=7, shuffle=True, random_state=seed)
            svm = SVC(kernel='rbf', random_state=seed)
            grid = GridSearchCV(
                svm, param_grid, cv=cv,
                scoring='f1_macro',  # optimize macro F1
                n_jobs=-1
            )
            grid.fit(X_pca, y)
            results.append((n_zones, n_comp, seed, grid.best_params_, grid.best_score_))

# --- Aggregate results across seeds ---
config_scores = defaultdict(list)
for n_zones, n_comp, seed, params, f1 in results:
    key = (n_zones, n_comp, str(params))
    config_scores[key].append(f1)

# --- Compute mean ± std for each configuration ---
config_summary = []
for (n_zones, n_comp, params_str), f1_list in config_scores.items():
    mean_f1 = np.mean(f1_list)
    std_f1 = np.std(f1_list)
    config_summary.append((mean_f1, std_f1, n_zones, n_comp, params_str, len(f1_list)))

# --- Sort by mean F1 descending ---
config_summary.sort(reverse=True, key=lambda x: x[0])

# --- Print top 5 configurations ---
print("\n=== Top SVM (RBF) Configurations (Aggregated over seeds) ===")
for mean_f1, std_f1, n_zones, n_comp, params_str, n_runs in config_summary[:10]:
    print(f"Top {n_zones:2d} zones | PCA {n_comp} | F1-macro = {mean_f1:.3f} ± {std_f1:.3f} "
          f"({n_runs} seeds) | Params: {params_str}")

# --- Most frequent best parameter sets ---
best_params_all = [r[3] for r in results]
param_counts = Counter(map(str, best_params_all))

print("\nParameter frequency across all configurations:")
for p, count in param_counts.most_common():
    print(f"{count} occurrences → {p}")


In [ ]:
# === SVM (RBF) PERFORMANCE EVALUATION – FINAL FIXED VERSION (CLASES 1 Y 2) ===

from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.svm import SVC
from sklearn.decomposition import PCA
from sklearn.metrics import (
    confusion_matrix, classification_report, roc_auc_score,
    average_precision_score, precision_recall_curve, roc_curve
)
import numpy as np
import matplotlib.pyplot as plt

# --- Data ---
kw_top_zones = fisher_df.head(2).index
X = df[kw_top_zones].values
y = phenotype.values    # valores 1 y 2

# --- PCA ---
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X)

# --- Best SVM parameters ---
svm_params = {
    'C': 10,
    'gamma': 'scale',
    'kernel': 'rbf',
    'probability': True
}

# --- Fit once to know class order ---
tmp_model = SVC(**svm_params)
tmp_model.fit(X_pca, y)
idx_class2 = np.where(tmp_model.classes_ == 2)[0][0]

# --- Initialize accumulators ---
n_runs = 50
accs, aucs, pr_aucs, f1_macros, f1_weighted = [], [], [], [], []
cm_total = np.zeros((2, 2))
phen2_precision, phen2_recall, phen2_f1 = [], [], []

# --- Collect ROC/PR data ---
fpr_all, tpr_all, prec_all, rec_all = [], [], [], []

# --- Loop over seeds ---
for seed in range(n_runs):
    cv = StratifiedKFold(n_splits=7, shuffle=True, random_state=seed)
    svm = SVC(**svm_params)

    y_pred = cross_val_predict(svm, X_pca, y, cv=cv)
    y_prob = cross_val_predict(svm, X_pca, y, cv=cv, method='predict_proba')
    y_prob_phen2 = y_prob[:, idx_class2]

    # Metrics
    accs.append(np.mean(y_pred == y))
    aucs.append(roc_auc_score(y, y_prob_phen2))        # sin labels
    pr_aucs.append(average_precision_score(y, y_prob_phen2, pos_label=2))
    cm_total += confusion_matrix(y, y_pred, labels=np.unique(y))

    report = classification_report(y, y_pred, output_dict=True)
    f1_macros.append(report['macro avg']['f1-score'])
    f1_weighted.append(report['weighted avg']['f1-score'])
    phen2_precision.append(report['2']['precision'])
    phen2_recall.append(report['2']['recall'])
    phen2_f1.append(report['2']['f1-score'])

    # ROC & PR data
    fpr, tpr, _ = roc_curve(y, y_prob_phen2, pos_label=2)
    prec, rec, _ = precision_recall_curve(y, y_prob_phen2, pos_label=2)

    fpr_all.append(np.interp(np.linspace(0,1,100), fpr, tpr))
    tpr_all.append(np.interp(np.linspace(0,1,100), fpr, tpr))
    prec_all.append(np.interp(np.linspace(0,1,100), rec[::-1], prec[::-1]))
    rec_all.append(np.linspace(0,1,100))

# --- Summary ---
print("\n=== SVM (RBF) Extended Evaluation (Cross-Validated) ===")
print(f"Accuracy: {np.mean(accs):.3f} ± {np.std(accs):.3f}")
print(f"ROC AUC: {np.mean(aucs):.3f} ± {np.std(aucs):.3f}")
print(f"PR AUC: {np.mean(pr_aucs):.3f} ± {np.std(pr_aucs):.3f}")
print(f"F1 Macro: {np.mean(f1_macros):.3f} ± {np.std(f1_macros):.3f}")
print(f"F1 Weighted: {np.mean(f1_weighted):.3f} ± {np.std(f1_weighted):.3f}")
print(f"Phenotype 2 → Precision: {np.mean(phen2_precision):.3f}, "
      f"Recall: {np.mean(phen2_recall):.3f}, F1: {np.mean(phen2_f1):.3f}")

print("\nMean confusion matrix:\n", cm_total / n_runs)

# --- Mean ROC/PR curves ---
mean_tpr = np.mean(tpr_all, axis=0)
mean_prec = np.mean(prec_all, axis=0)
mean_rec = np.mean(rec_all, axis=0)

results['SVM (RBF)'] = {
    'tpr': mean_tpr,
    'auc': np.mean(aucs),
    'prec': mean_prec,
    'rec': mean_rec,
    'ap': np.mean(pr_aucs)
}

plt.figure(figsize=(12,5))

plt.subplot(1,2,1)
plt.plot(np.linspace(0,1,100), mean_tpr, label=f"Mean AUC = {np.mean(aucs):.3f}")
plt.plot([0,1],[0,1],'--',color='gray')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Mean ROC Curve (Phenotype 2)")
plt.legend()

plt.subplot(1,2,2)
plt.plot(mean_rec, mean_prec, label=f"Mean AP = {np.mean(pr_aucs):.3f}")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Mean Precision–Recall Curve (Phenotype 2)")
plt.legend()

plt.tight_layout()
plt.show()


In [ ]:
# === SVM (RBF) BOOTSTRAP EVALUATION – FIXED & STABILIZED (CLASES 1 Y 2) ===

from sklearn.utils import resample
from sklearn.svm import SVC
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, average_precision_score
import numpy as np

# --- Data ---
kw_top_zones = fisher_df.head(2).index 
X = df[kw_top_zones].values
y = phenotype.values   # ahora valores 1 y 2

# --- PCA ---
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X)

# --- Best SVM parameters ---
svm_params = {
    'C': 10,
    'gamma': 'scale',
    'kernel': 'rbf',
    'probability': True
}

# --- Bootstrap ---
n_bootstrap = 1000
min_oob_size = 5
accs_bs, f1_macro_bs, f1_weighted_bs, auc_bs, pr_auc_bs = [], [], [], [], []

# Fit once to get class index
tmp_model = SVC(**svm_params)
tmp_model.fit(X_pca, y)
idx_class2 = np.where(tmp_model.classes_ == 2)[0][0]  # ahora clase positiva = 2

for i in range(n_bootstrap):
    # Resample training indices
    train_idx = np.unique(resample(np.arange(len(X_pca)), replace=True, stratify=y))
    X_res, y_res = X_pca[train_idx], y[train_idx]

    # Fit model on bootstrap sample
    model = SVC(**svm_params)
    model.fit(X_res, y_res)

    # OOB = complement of training indices
    oob_mask = ~np.isin(np.arange(len(X_pca)), train_idx)
    X_oob = X_pca[oob_mask]
    y_oob = y[oob_mask]

    # Skip if OOB too small or only one class
    if len(y_oob) < min_oob_size:
        continue
    if len(np.unique(y_oob)) < 2:
        continue

    # Evaluate on OOB
    y_pred = model.predict(X_oob)
    y_prob = model.predict_proba(X_oob)[:, idx_class2]

    accs_bs.append(accuracy_score(y_oob, y_pred))
    f1_macro_bs.append(f1_score(y_oob, y_pred, average='macro'))
    f1_weighted_bs.append(f1_score(y_oob, y_pred, average='weighted'))
    auc_bs.append(roc_auc_score(y_oob, y_prob, labels=[1, 2]))
    pr_auc_bs.append(average_precision_score(y_oob, y_prob, pos_label=2))

# --- Compute 95% confidence intervals ---
def ci95(arr):
    return np.percentile(arr, 2.5), np.percentile(arr, 97.5)

print("=== SVM (RBF) Metrics with 95% CI (OOB, Stabilized) ===")
print(f"Accuracy: {np.mean(accs_bs):.3f} (95% CI {ci95(accs_bs)})")
print(f"F1 Macro: {np.mean(f1_macro_bs):.3f} (95% CI {ci95(f1_macro_bs)})")
print(f"F1 Weighted: {np.mean(f1_weighted_bs):.3f} (95% CI {ci95(f1_weighted_bs)})")
print(f"ROC AUC: {np.mean(auc_bs):.3f} (95% CI {ci95(auc_bs)})")
print(f"PR AUC: {np.mean(pr_auc_bs):.3f} (95% CI {ci95(pr_auc_bs)})")


## Random Forest

In [ ]:
# === GRID SEARCH FOR RANDOM FOREST (MULTI-ZONE, MULTI-PCA, MULTI-SEED) ===

from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.decomposition import PCA
from collections import Counter, defaultdict
import numpy as np

# --- Parameter grid for Random Forest ---
param_grid = {
    'n_estimators': [10],
    'min_samples_split': [4, 8],
    'min_samples_leaf': [2, 3, 5],
    'max_features': ['sqrt', 'log2', None],
    'class_weight': [{2: 1, 3: 1.2}]
}

# --- Range of top Kruskal–Wallis zones and PCA components ---
zone_range = range(2, 5)   # top 18–28 zones
pca_range = range(2, 3)      # PCA components: 2–4

# --- Store results ---
results = []

for n_zones in zone_range:
    kw_top_zones = combined_df.head(n_zones).index
    X = df[kw_top_zones].values
    y = phenotype.values

    for n_comp in pca_range:
        # --- PCA ---
        pca = PCA(n_components=n_comp)
        X_pca = pca.fit_transform(X)

        for seed in range(30):  # number of random seeds
            cv = StratifiedKFold(n_splits=7, shuffle=True, random_state=seed)
            rf = RandomForestClassifier(random_state=seed)
            grid = GridSearchCV(
                rf, param_grid, cv=cv,
                scoring='f1_macro', 
                n_jobs=-1
            )
            grid.fit(X_pca, y)
            results.append((n_zones, n_comp, seed, grid.best_params_, grid.best_score_))

# --- Aggregate results across seeds ---
config_scores = defaultdict(list)
for n_zones, n_comp, seed, params, acc in results:
    key = (n_zones, n_comp, str(params))
    config_scores[key].append(acc)

# --- Compute mean ± std for each configuration ---
config_summary = []
for (n_zones, n_comp, params_str), acc_list in config_scores.items():
    mean_acc = np.mean(acc_list)
    std_acc = np.std(acc_list)
    config_summary.append((mean_acc, std_acc, n_zones, n_comp, params_str, len(acc_list)))

# --- Sort by mean accuracy descending ---
config_summary.sort(reverse=True, key=lambda x: x[0])

# --- Print top 5 configurations ---
print("\n=== Top Random Forest Configurations (Aggregated over seeds) ===")
for mean_acc, std_acc, n_zones, n_comp, params_str, n_runs in config_summary[:10]:
    print(f"Top {n_zones:2d} zones | PCA {n_comp} | Accuracy = {mean_acc:.3f} ± {std_acc:.3f} "
          f"({n_runs} seeds) | Params: {params_str}")

# --- Most frequent best parameter sets ---
best_params_all = [r[3] for r in results]
param_counts = Counter(map(str, best_params_all))

print("\nParameter frequency across all configurations:")
for p, count in param_counts.most_common():
    print(f"{count} occurrences → {p}")


In [ ]:
# === RANDOM FOREST PERFORMANCE EVALUATION (CLASES 1 Y 2) ===

from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.ensemble import RandomForestClassifier
from sklearn.decomposition import PCA
from sklearn.metrics import (
    confusion_matrix, classification_report, roc_auc_score,
    average_precision_score, precision_recall_curve, roc_curve
)
import numpy as np
import matplotlib.pyplot as plt

# --- Data ---
kw_top_zones = fisher_df.head(2).index
X = df[kw_top_zones].values
y = phenotype.values  # ahora valores 1 y 2

# --- PCA ---
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X)

# --- Best Random Forest parameters from previous grid search ---
rf_params = {
    'n_estimators': 20,
    'min_samples_split': 4,
    'min_samples_leaf': 2,
    'max_features': None
}

# --- Fit once to know class order ---
tmp_model = RandomForestClassifier(**rf_params)
tmp_model.fit(X_pca, y)
idx_class2 = np.where(tmp_model.classes_ == 2)[0][0]  # ahora clase positiva = 2

# --- Initialize accumulators ---
n_runs = 50
accs, aucs, pr_aucs, f1_macros, f1_weighted = [], [], [], [], []
cm_total = np.zeros((2, 2))
phen2_precision, phen2_recall, phen2_f1 = [], [], []

# --- Collect ROC/PR data for averaged curves ---
fpr_all, tpr_all, prec_all, rec_all = [], [], [], []

# --- Loop over random seeds ---
for seed in range(n_runs):
    cv = StratifiedKFold(n_splits=7, shuffle=True, random_state=seed)
    rf = RandomForestClassifier(**rf_params, random_state=seed)

    # Cross-validated predictions (out-of-fold)
    y_pred = cross_val_predict(rf, X_pca, y, cv=cv)
    y_prob = cross_val_predict(rf, X_pca, y, cv=cv, method='predict_proba')
    y_prob_phen2 = y_prob[:, idx_class2]

    # --- Metrics per run ---
    accs.append(np.mean(y_pred == y))
    aucs.append(roc_auc_score(y, y_prob_phen2, labels=[1, 2]))
    pr_aucs.append(average_precision_score(y, y_prob_phen2, pos_label=2))
    cm_total += confusion_matrix(y, y_pred, labels=np.unique(y))

    report = classification_report(y, y_pred, output_dict=True)
    f1_macros.append(report['macro avg']['f1-score'])
    f1_weighted.append(report['weighted avg']['f1-score'])
    phen2_precision.append(report['2']['precision'])
    phen2_recall.append(report['2']['recall'])
    phen2_f1.append(report['2']['f1-score'])

    # --- ROC & PR data for averaging ---
    fpr, tpr, _ = roc_curve(y, y_prob_phen2, pos_label=2)
    prec, rec, _ = precision_recall_curve(y, y_prob_phen2, pos_label=2)
    fpr_all.append(np.interp(np.linspace(0,1,100), fpr, tpr))
    tpr_all.append(np.interp(np.linspace(0,1,100), fpr, tpr))
    prec_all.append(np.interp(np.linspace(0,1,100), rec[::-1], prec[::-1]))
    rec_all.append(np.linspace(0,1,100))

# --- Summary ---
print("\n=== Random Forest Extended Evaluation (Cross-Validated) ===")
print(f"Accuracy: {np.mean(accs):.3f} ± {np.std(accs):.3f}")
print(f"ROC AUC: {np.mean(aucs):.3f} ± {np.std(aucs):.3f}")
print(f"PR AUC: {np.mean(pr_aucs):.3f} ± {np.std(pr_aucs):.3f}")
print(f"F1 Macro: {np.mean(f1_macros):.3f} ± {np.std(f1_macros):.3f}")
print(f"F1 Weighted: {np.mean(f1_weighted):.3f} ± {np.std(f1_weighted):.3f}")
print(f"Phenotype 2 → Precision: {np.mean(phen2_precision):.3f}, "
      f"Recall: {np.mean(phen2_recall):.3f}, F1: {np.mean(phen2_f1):.3f}")

print("\nMean confusion matrix:\n", cm_total / n_runs)

# --- Mean ROC and PR curves across seeds ---
mean_tpr = np.mean(tpr_all, axis=0)
mean_prec = np.mean(prec_all, axis=0)
mean_rec = np.mean(rec_all, axis=0)

results['Random Forest'] = {
    'tpr': np.mean(tpr_all, axis=0),
    'auc': np.mean(aucs),
    'prec': np.mean(prec_all, axis=0),
    'rec': np.mean(rec_all, axis=0),
    'ap': np.mean(pr_aucs)
}

plt.figure(figsize=(12,5))

plt.subplot(1,2,1)
plt.plot(np.linspace(0,1,100), mean_tpr, label=f"Mean AUC = {np.mean(aucs):.3f}")
plt.plot([0,1],[0,1],'--',color='gray')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Mean ROC Curve (Phenotype 2)")
plt.legend()

plt.subplot(1,2,2)
plt.plot(mean_rec, mean_prec, label=f"Mean AP = {np.mean(pr_aucs):.3f}")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Mean Precision–Recall Curve (Phenotype 2)")
plt.legend()

plt.tight_layout()
plt.show()


In [ ]:
# === RANDOM FOREST BOOTSTRAP EVALUATION – FIXED & STABILIZED (CLASES 1 Y 2) ===

from sklearn.utils import resample
from sklearn.ensemble import RandomForestClassifier
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, average_precision_score
import numpy as np

# --- Data ---
kw_top_zones = fisher_df.head(2).index
X = df[kw_top_zones].values
y = phenotype.values  # ahora valores 1 y 2

# --- PCA ---
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X)

# --- Best Random Forest parameters ---
rf_params = {
    'n_estimators': 20,
    'min_samples_split': 4,
    'min_samples_leaf': 2,
    'max_features': None
}

# --- Bootstrap ---
n_bootstrap = 1000
min_oob_size = 5  # skip iterations with very small OOB
accs_bs, f1_macro_bs, f1_weighted_bs, auc_bs, pr_auc_bs = [], [], [], [], []

# Fit once to get class index
tmp_model = RandomForestClassifier(**rf_params, random_state=0)
tmp_model.fit(X_pca, y)
idx_class2 = np.where(tmp_model.classes_ == 2)[0][0]  # ahora clase positiva = 2

for i in range(n_bootstrap):
    # Stratified resample for training indices
    train_idx = np.unique(resample(np.arange(len(X_pca)), replace=True, stratify=y))
    X_res, y_res = X_pca[train_idx], y[train_idx]

    # Fit model on bootstrap sample
    model = RandomForestClassifier(**rf_params, random_state=i)
    model.fit(X_res, y_res)

    # OOB = complement of training indices
    oob_mask = ~np.isin(np.arange(len(X_pca)), train_idx)
    X_oob = X_pca[oob_mask]
    y_oob = y[oob_mask]

    # Skip if OOB too small or only one class
    if len(y_oob) < min_oob_size:
        continue
    if len(np.unique(y_oob)) < 2:
        continue

    # Evaluate on OOB
    y_pred = model.predict(X_oob)
    y_prob = model.predict_proba(X_oob)[:, idx_class2]

    accs_bs.append(accuracy_score(y_oob, y_pred))
    f1_macro_bs.append(f1_score(y_oob, y_pred, average='macro'))
    f1_weighted_bs.append(f1_score(y_oob, y_pred, average='weighted'))
    auc_bs.append(roc_auc_score(y_oob, y_prob, labels=[1, 2]))
    pr_auc_bs.append(average_precision_score(y_oob, y_prob, pos_label=2))

# --- Compute 95% confidence intervals ---
def ci95(arr):
    return np.percentile(arr, 2.5), np.percentile(arr, 97.5)

print("=== Random Forest Metrics with 95% CI (OOB, Stabilized) ===")
print(f"Accuracy: {np.mean(accs_bs):.3f} (95% CI {ci95(accs_bs)})")
print(f"F1 Macro: {np.mean(f1_macro_bs):.3f} (95% CI {ci95(f1_macro_bs)})")
print(f"F1 Weighted: {np.mean(f1_weighted_bs):.3f} (95% CI {ci95(f1_weighted_bs)})")
print(f"ROC AUC: {np.mean(auc_bs):.3f} (95% CI {ci95(auc_bs)})")
print(f"PR AUC: {np.mean(pr_auc_bs):.3f} (95% CI {ci95(pr_auc_bs)})")


## Logistic regression

In [ ]:
# === GRID SEARCH FOR LOGISTIC REGRESSION (balanced scoring, robust results) ===

from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA
from collections import Counter, defaultdict
import numpy as np

# --- Parameter grid for Logistic Regression ---
param_grid = {
    'C': np.linspace(4, 5, 10),
    'penalty': ['l1', 'l2'],
    'solver': ['liblinear', 'saga'],
    'max_iter': [1000],
    'class_weight': [{2: 1, 3: 1.2}]
}

#np.logspace(-2, 2, 5)
#np.linspace(0.05, 0.3, 6)

# --- Range of top Kruskal–Wallis zones to test ---
zone_range = range(2, 4)  # from 18 to 24 inclusive

# --- Loop over zone sizes and random seeds ---
results = []

for n_zones in zone_range:
    kw_top_zones = fisher_df.head(n_zones).index
    X = df[kw_top_zones].values
    y = phenotype.values

    # --- PCA (optional) ---
    pca = PCA(n_components=2)
    X_pca = pca.fit_transform(X)

    for seed in range(50):  # can increase to 50 if needed
        cv = StratifiedKFold(n_splits=7, shuffle=True, random_state=seed)
        logreg = LogisticRegression(random_state=seed)
        grid = GridSearchCV(
            logreg, param_grid, cv=cv,
            scoring='f1_macro',    # optimize macro F1
            n_jobs=-1
        )
        grid.fit(X_pca, y)
        results.append((n_zones, seed, grid.best_params_, grid.best_score_))

# --- Aggregate results across seeds ---
config_scores = defaultdict(list)
for n_zones, seed, params, f1 in results:
    key = (n_zones, str(params))  # string because dicts are unhashable
    config_scores[key].append(f1)

# --- Compute mean ± std for each configuration ---
config_summary = []
for (n_zones, params_str), f1_list in config_scores.items():
    mean_f1 = np.mean(f1_list)
    std_f1 = np.std(f1_list)
    config_summary.append((mean_f1, std_f1, n_zones, params_str, len(f1_list)))

# --- Sort by mean F1 descending ---
config_summary.sort(reverse=True, key=lambda x: x[0])

# --- Print top 5 configurations ---
print("\n=== Top Logistic Regression Configurations (Aggregated over seeds) ===")
for mean_f1, std_f1, n_zones, params_str, n_runs in config_summary[:10]:
    print(f"Top {n_zones:2d} zones | F1-macro = {mean_f1:.3f} ± {std_f1:.3f} "
          f"({n_runs} seeds) | Params: {params_str}")

# --- Most frequent parameter combinations overall ---
best_params_all = [r[2] for r in results]
param_counts = Counter(map(str, best_params_all))

print("\nParameter frequency across all zone sizes and seeds:")
for p, count in param_counts.most_common():
    print(f"{count} occurrences → {p}")


In [ ]:
# === LOGISTIC REGRESSION PERFORMANCE EVALUATION – FINAL FIXED VERSION (CV REPEATED, CLASES 1 Y 2) ===

from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA
from sklearn.metrics import (
    confusion_matrix, classification_report, roc_auc_score,
    average_precision_score, precision_recall_curve, roc_curve
)
import numpy as np
import matplotlib.pyplot as plt

# --- Data ---
kw_top_zones = fisher_df.head(2).index
X = df[kw_top_zones].values
y = phenotype.values  # ahora valores 1 y 2

# --- PCA ---
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X)

# --- Best parameters from grid search ---
logreg_params = {
    'C': 4.4,
    'penalty': 'l2',
    'solver': 'liblinear',
    'max_iter': 1000
}

# --- Fit once to know class order ---
tmp_model = LogisticRegression(**logreg_params)
tmp_model.fit(X_pca, y)
idx_class2 = np.where(tmp_model.classes_ == 2)[0][0]  # ahora clase positiva = 2

# --- Initialize accumulators ---
n_runs = 50
accs, aucs, pr_aucs, f1_macros, f1_weighted = [], [], [], [], []
cm_total = np.zeros((2, 2))
phen2_precision, phen2_recall, phen2_f1 = [], [], []

# --- Collect ROC/PR data for averaged curves ---
fpr_all, tpr_all, prec_all, rec_all = [], [], [], []

# --- Loop over random seeds ---
for seed in range(n_runs):
    cv = StratifiedKFold(n_splits=7, shuffle=True, random_state=seed)
    logreg = LogisticRegression(**logreg_params, random_state=seed)
    
    # Cross-validated predictions (out-of-fold)
    y_pred = cross_val_predict(logreg, X_pca, y, cv=cv)
    y_prob = cross_val_predict(logreg, X_pca, y, cv=cv, method='predict_proba')
    y_prob_phen2 = y_prob[:, idx_class2]
    
    # --- Metrics per run ---
    accs.append(np.mean(y_pred == y))
    aucs.append(roc_auc_score(y, y_prob_phen2, labels=[1, 2]))
    pr_aucs.append(average_precision_score(y, y_prob_phen2, pos_label=2))
    cm_total += confusion_matrix(y, y_pred, labels=np.unique(y))
    
    report = classification_report(y, y_pred, output_dict=True)
    f1_macros.append(report['macro avg']['f1-score'])
    f1_weighted.append(report['weighted avg']['f1-score'])
    phen2_precision.append(report['2']['precision'])
    phen2_recall.append(report['2']['recall'])
    phen2_f1.append(report['2']['f1-score'])
    
    # --- ROC & PR data for averaging ---
    fpr, tpr, _ = roc_curve(y, y_prob_phen2, pos_label=2)
    prec, rec, _ = precision_recall_curve(y, y_prob_phen2, pos_label=2)
    fpr_all.append(np.interp(np.linspace(0,1,100), fpr, tpr))
    tpr_all.append(np.interp(np.linspace(0,1,100), fpr, tpr))
    prec_all.append(np.interp(np.linspace(0,1,100), rec[::-1], prec[::-1]))  # reversed for interp
    rec_all.append(np.linspace(0,1,100))

# --- Summary ---
print("\n=== Logistic Regression (Extended Evaluation) ===")
print(f"Accuracy: {np.mean(accs):.3f} ± {np.std(accs):.3f}")
print(f"ROC AUC: {np.mean(aucs):.3f} ± {np.std(aucs):.3f}")
print(f"PR AUC: {np.mean(pr_aucs):.3f} ± {np.std(pr_aucs):.3f}")
print(f"F1 Macro: {np.mean(f1_macros):.3f} ± {np.std(f1_macros):.3f}")
print(f"F1 Weighted: {np.mean(f1_weighted):.3f} ± {np.std(f1_weighted):.3f}")
print(f"Phenotype 2 → Precision: {np.mean(phen2_precision):.3f}, "
      f"Recall: {np.mean(phen2_recall):.3f}, F1: {np.mean(phen2_f1):.3f}")

print("\nMean confusion matrix:\n", cm_total / n_runs)

# --- Mean ROC and PR curves across seeds ---
mean_tpr = np.mean(tpr_all, axis=0)
mean_prec = np.mean(prec_all, axis=0)
mean_rec = np.mean(rec_all, axis=0)

results['Logistic Regression'] = {
    'tpr': np.mean(tpr_all, axis=0),
    'auc': np.mean(aucs),
    'prec': np.mean(prec_all, axis=0),
    'rec': np.mean(rec_all, axis=0),
    'ap': np.mean(pr_aucs)
}

plt.figure(figsize=(12,5))

plt.subplot(1,2,1)
plt.plot(np.linspace(0,1,100), mean_tpr, label=f"Mean AUC = {np.mean(aucs):.3f}")
plt.plot([0,1],[0,1],'--',color='gray')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Mean ROC Curve (Phenotype 2)")
plt.legend()

plt.subplot(1,2,2)
plt.plot(mean_rec, mean_prec, label=f"Mean AP = {np.mean(pr_aucs):.3f}")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Mean Precision–Recall Curve (Phenotype 2)")
plt.legend()

plt.tight_layout()
plt.show()


In [ ]:
# === LOGISTIC REGRESSION BOOTSTRAP EVALUATION – FIXED & STABILIZED (CLASES 1 Y 2) ===

from sklearn.utils import resample
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, average_precision_score
import numpy as np

# --- Data ---
kw_top_zones = fisher_df.head(2).index
X = df[kw_top_zones].values
y = phenotype.values  # ahora valores 1 y 2

# --- PCA ---
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X)

# --- Best Logistic Regression parameters ---
logreg_params = {
    'C': 4.4,
    'penalty': 'l2',
    'solver': 'liblinear',
    'max_iter': 1000
}

# --- Bootstrap ---
n_bootstrap = 1000
min_oob_size = 5  # skip iterations with very small OOB
accs_bs, f1_macro_bs, f1_weighted_bs, auc_bs, pr_auc_bs = [], [], [], [], []

# Fit once to get class index
tmp_model = LogisticRegression(**logreg_params)
tmp_model.fit(X_pca, y)
idx_class2 = np.where(tmp_model.classes_ == 2)[0][0]  # ahora clase positiva = 2

for i in range(n_bootstrap):
    # Stratified resample for training indices
    train_idx = np.unique(resample(np.arange(len(X_pca)), replace=True, stratify=y))
    X_res, y_res = X_pca[train_idx], y[train_idx]

    # Fit model on bootstrap sample
    model = LogisticRegression(**logreg_params)
    model.fit(X_res, y_res)

    # OOB = complement of training indices
    oob_mask = ~np.isin(np.arange(len(X_pca)), train_idx)
    X_oob = X_pca[oob_mask]
    y_oob = y[oob_mask]

    # Skip if OOB too small or only one class
    if len(y_oob) < min_oob_size:
        continue
    if len(np.unique(y_oob)) < 2:
        continue

    # Evaluate on OOB
    y_pred = model.predict(X_oob)
    y_prob = model.predict_proba(X_oob)[:, idx_class2]

    accs_bs.append(accuracy_score(y_oob, y_pred))
    f1_macro_bs.append(f1_score(y_oob, y_pred, average='macro'))
    f1_weighted_bs.append(f1_score(y_oob, y_pred, average='weighted'))
    auc_bs.append(roc_auc_score(y_oob, y_prob, labels=[1, 2]))
    pr_auc_bs.append(average_precision_score(y_oob, y_prob, pos_label=2))

# --- Compute 95% confidence intervals ---
def ci95(arr):
    return np.percentile(arr, 2.5), np.percentile(arr, 97.5)

print("=== Logistic Regression Metrics with 95% CI (OOB, Stabilized) ===")
print(f"Accuracy: {np.mean(accs_bs):.3f} (95% CI {ci95(accs_bs)})")
print(f"F1 Macro: {np.mean(f1_macro_bs):.3f} (95% CI {ci95(f1_macro_bs)})")
print(f"F1 Weighted: {np.mean(f1_weighted_bs):.3f} (95% CI {ci95(f1_weighted_bs)})")
print(f"ROC AUC: {np.mean(auc_bs):.3f} (95% CI {ci95(auc_bs)})")
print(f"PR AUC: {np.mean(pr_auc_bs):.3f} (95% CI {ci95(pr_auc_bs)})")


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fpr_grid = np.linspace(0, 1, 100)
plt.figure(figsize=(12,5))

# --- Curvas ROC ---
plt.subplot(1,2,1)
for model, data in results.items():
    if isinstance(data, dict) and 'tpr' in data and len(data['tpr']) == len(fpr_grid):
        plt.plot(fpr_grid, data['tpr'], label=f"{model} (AUC = {data['auc']:.3f})")
plt.plot([0,1], [0,1], '--', color='gray')
plt.xlabel("Tasa de Falsos Positivos")
plt.ylabel("Tasa de Verdaderos Positivos")
plt.title("Curvas ROC Promedio entre Modelos")
plt.legend()
plt.grid(alpha=0.3)

# --- Curvas PR ---
plt.subplot(1,2,2)
for model, data in results.items():
    if isinstance(data, dict) and 'prec' in data and 'rec' in data and len(data['prec']) == len(fpr_grid):
        plt.plot(data['rec'], data['prec'], label=f"{model} (AP = {data['ap']:.3f})")
plt.xlabel("Recall")
plt.ylabel("Precisión")
plt.title("Curvas Precisión–Recall Promedio entre Modelos")
plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("ROC_PR_all_modelspn.png", dpi=300)
plt.show()
